# Semana 9: RQ4, máscaras de referência e consenso

Seções 1 a 3 de `prereg/plano_rq4.md` e as mudanças datadas no fim dele. **Este notebook não roda nenhum modelo.** Ele junta as referências do dia 06/05/2024 (camada Planet/SkySat da UFRGS e os cinco produtos de graduação do EMSR720), a água permanente (JRC GSW ≥ 90%), o WorldCover 2021 e as máscaras de entrada do Sentinel-2 (sem dado, nuvem e SCL), e grava para cada um dos três tiles (22JDM, 22JCN, 22JDN) um código por pixel dizendo se ele é pontuado (água ou terra no consenso) ou por que não é.

| Código | Significado |
|---|---|
| 0, 1 | terra, água (pontuados) |
| 2 | sem dado de entrada (seis bandas iguais a 0) |
| 3 | fora de toda referência |
| 4 | classe não pontuada (rastro de inundação ou deslizamento do EMSR720) |
| 5 | as referências que cobrem o pixel discordam |
| 6 | a até 2 pixels de uma fronteira água-terra do consenso (regra `borda` da semana 6) |
| 7 | nuvem (s2cloudless ≥ 40%) |
| 8 | SCL 3, 8, 9 ou 10 |
| 9 | sem probabilidade de nuvem ou sem SCL |

Quando vale mais de um motivo, fica o primeiro da tabela; a contagem de cada motivo, com sobreposição, também é gravada. A área mapeada da camada da UFRGS é a faixa de 1 km em volta da mancha (principal), com as sensibilidades de 500 m e 2 km; a do EMSR720 é a área de interesse dentro da pegada da imagem em que o dano foi mapeado, menos o que o produto marca como não analisado.

| Arquivo, em `dados/rs2024/mascaras` | Conteúdo |
|---|---|
| `<tile>_classe_1000m.tif` (e `_500m`, `_2000m`) | banda 1: código acima; banda 2: número de referências que cobrem o pixel |
| `<tile>_estrato.tif` | 1 construído (WorldCover 50), 2 água permanente (WorldCover 80), 3 outro |

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, clona o repositório, lê os registros das entradas |
| 2 | Lê e confere as referências (SHA-256 dos pacotes, classes, fonte de cada produto) |
| 3 | Monta as máscaras dos três tiles no disco local, com a conferência de alinhamento e as janelas de 512 × 512 (15 a 30 min) |
| 4 | Copia para o Drive sem sobrescrever, remonta o Drive, confere os SHA-256 e grava o registro |

Abra o Colab logada na conta do mestrado, sem GPU (se a RAM acabar, use o ambiente de RAM alta). Se alguma célula der erro, pare e guarde a saída inteira. A saída de cada rodada vai para uma pasta nova (`rq4_mascaras`, depois `rq4_mascaras_2`, e assim por diante).

A conferência de alinhamento compara a mancha de cada referência, deslocada de −3 a +3 pixels, com a água lida da própria imagem (B3 > B8A), perto da borda da mancha. Ela não usa o modelo e só procura um erro de grade: o melhor deslocamento deve ser (0, 0) ou vizinho dele. Onde a água já tinha baixado na hora da imagem, a borda da referência não coincide com a da água e a concordância quase não muda com o deslocamento; por isso a célula só avisa quando um deslocamento maior que 1 pixel ganha mais de 0,01 de concordância sobre (0, 0). A água permanente não entra na conferência: ela vem do Earth Engine na mesma grade do Sentinel-2, e durante a cheia a sua borda fica dentro da água.

Arquivos a conferir depois da célula 4, na pasta de saída: `rq4_mascaras.json`, `rq4_mascaras_sha256.csv`, `rq4_janelas.csv`, `emsr720_fontes.csv` e as três figuras `<tile>_mascara.png`.

In [ ]:
# Célula 1 de 4: Drive, repositório e registros
import os, re, json, time, hashlib, glob, shutil, subprocess, sys, csv, zipfile
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio', 'pyogrio'], check=True)
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

S2DIR = f'{RAIZ}/dados/rs2024/s2'
AUXDIR = f'{RAIZ}/dados/rs2024/aux'
REF = f'{RAIZ}/dados/rs2024/referencias'
DESTINO = f'{RAIZ}/dados/rs2024/mascaras'
SAIDA_BASE = f'{RAIZ}/resultados/semana9/rq4_mascaras'
SAIDA_DRIVE, n = SAIDA_BASE, 1
while os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE):      # nunca sobrescreve
    n += 1
    SAIDA_DRIVE = f'{SAIDA_BASE}_{n}'
print('Saída desta rodada:', SAIDA_DRIVE.replace(MEUDRIVE, 'MyDrive'))
LOCAL = '/content/rq4_mascaras'
ENTRADAS_LOCAL = f'{LOCAL}/entradas'
SAIDA_LOCAL = f'{LOCAL}/saida'
for d in (DESTINO, ENTRADAS_LOCAL, SAIDA_LOCAL):
    os.makedirs(d, exist_ok=True)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
import s2_exportacao as S
import mascaras as M
SHA_MODULO = hashlib.sha256(open(f'{REPO_GIT}/analysis/rq4/mascaras.py', 'rb').read()).hexdigest()

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def esperar(caminho, limite=180):
    t0 = time.time()
    while not os.path.exists(caminho):
        if time.time() - t0 > limite:
            raise FileNotFoundError(f'{caminho} não apareceu no Drive em {limite} s. Pare e guarde esta saída.')
        time.sleep(5)

def trazer(no_drive, sha_esperado, pasta=ENTRADAS_LOCAL):
    # Copia do Drive para o disco local e confere o SHA-256 com o registro do repositório.
    # Se não conferir, remonta o Drive uma vez e copia de novo (o Drive às vezes mostra uma versão velha do arquivo).
    local = f'{pasta}/{os.path.basename(no_drive)}'
    for tentativa in (1, 2):
        esperar(no_drive)
        if not (os.path.exists(local) and sha256_arquivo(local) == sha_esperado):
            shutil.copyfile(no_drive, local + '.parcial')
            os.replace(local + '.parcial', local)
        sha = sha256_arquivo(local)
        if sha == sha_esperado:
            return local
        if tentativa == 1:
            print(f'   {os.path.basename(no_drive)}: SHA-256 diferente do registro; remontando o Drive e copiando de novo.')
            drive.flush_and_unmount()
            time.sleep(10)
            drive.mount('/content/drive', force_remount=True)
    raise ValueError(f'{no_drive}: SHA-256 {sha[:16]}..., o registro diz {sha_esperado[:16]}... Pare e guarde esta saída.')

# Registros do repositório: entradas do Sentinel-2, camadas auxiliares e referências.
REG_S2 = json.load(open(f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas.json'))
SHA_S2 = {r['arquivo']: r['sha256'] for r in csv.DictReader(open(f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas_sha256.csv'))}
REG_AUX = json.load(open(f'{REPO_GIT}/analysis/rq4/entradas/rq4_auxiliares.json'))
REG_REF = json.load(open(f'{REPO_GIT}/analysis/rq4/referencias/rodada.json'))
REG_EMSR = json.load(open(f'{REPO_GIT}/analysis/rq4/referencias/emsr720/rodada.json'))
INV = list(csv.DictReader(open(f'{REPO_GIT}/analysis/rq4/referencias/inventario_referencias.csv')))

GRADES = {}
for e in REG_S2['entradas']:
    g = e['grade']
    assert g['crs'] == S.CRS_ESPERADO[e['tile']] and (g['altura'], g['largura']) == (S.LADO, S.LADO), e['tile']
    assert REG_AUX['grades'][e['tile']]['transform'] == g['transform'], f"{e['tile']}: grade das camadas auxiliares diferente da do Sentinel-2"
    canon = {k: f'canonico/{e["tile"]}_{e["id"]}_{k}.tif' for k in ('s2', 'nuvem', 'scl')}
    assert all(c in SHA_S2 for c in canon.values()), canon
    GRADES[e['tile']] = {'crs': g['crs'], 'transform': tuple(g['transform']), 'canonico': canon,
                         'zeros': e['fracao_pixels_com_6_bandas_zero']}
assert list(GRADES) == list(S.TILES), list(GRADES)
M_CRS = 'EPSG:32722'
assert all(g['crs'] == M_CRS for g in GRADES.values())
print(f'Repositório no commit {commit[:7]}. Tiles:', {t: g['transform'][2:6:3] for t, g in GRADES.items()})
print('Distâncias da área mapeada da UFRGS (m):', M.DISTANCIAS_M)

In [ ]:
# Célula 2 de 4: referências
import warnings
warnings.filterwarnings('ignore', message=r'.*Measured \(M\) geometry.*')
import geopandas as gpd
import pyogrio

ENTRADAS_VERIFICADAS = []

# Camada Planet/SkySat da UFRGS (Zenodo 14617475, v1.4)
NOME_GPKG = 'ufrgs/cheias_rhguaiba_2024_db_v14.gpkg'
CAMADA_UFRGS = 'rhguaiba_planet-skysat_inundacao_obs_20240506'
gpkg = trazer(f'{REF}/{NOME_GPKG}', REG_REF['arquivos'][NOME_GPKG]['sha256'])
ENTRADAS_VERIFICADAS.append({'arquivo': f'referencias/{NOME_GPKG}', 'sha256': REG_REF['arquivos'][NOME_GPKG]['sha256']})
UFRGS = gpd.read_file(gpkg, layer=CAMADA_UFRGS).to_crs(M_CRS)
UFRGS = UFRGS[UFRGS.geometry.notna() & ~UFRGS.geometry.is_empty]
n_inv = [int(r['n_feicoes']) for r in INV if r['camada'] == CAMADA_UFRGS]
assert n_inv and len(UFRGS) == n_inv[0], f'{CAMADA_UFRGS}: {len(UFRGS)} feições, o inventário diz {n_inv}'
print(f'UFRGS {CAMADA_UFRGS}: {len(UFRGS)} feições, colunas {list(UFRGS.columns)}')

# Produtos de graduação do EMSR720 de 06/05/2024
PEDACOS = ('areaOfInterestA', 'imageFootprintA', 'observedEventA', 'source')
EXTRAIDO = f'{LOCAL}/emsr720'
EMSR, FONTES_LINHAS = {}, []
for p in sorted(REG_EMSR['produtos'], key=lambda p: p['aoi']):
    nome_zip = p['url'].rsplit('/', 1)[1]
    sha = REG_EMSR['arquivos_baixados'][nome_zip]['sha256']
    achados = sorted(set(glob.glob(f'{REF}/**/{nome_zip}', recursive=True)))
    assert achados, f'{nome_zip} não está em {REF}. Pare e guarde esta saída.'
    z = trazer(achados[0], sha)
    ENTRADAS_VERIFICADAS.append({'arquivo': os.path.relpath(achados[0], f'{RAIZ}/dados/rs2024'), 'sha256': sha})
    aoi = f"AOI{p['aoi']:02d}"
    destino = f'{EXTRAIDO}/{aoi}'
    os.makedirs(destino, exist_ok=True)
    with zipfile.ZipFile(z) as zz:
        for item in zz.infolist():
            base = os.path.basename(item.filename)
            if not item.is_dir() and any(f'_{k}_v' in base for k in PEDACOS) and base.lower().endswith(('.shp', '.shx', '.dbf', '.prj', '.cpg')):
                with zz.open(item) as fo, open(f'{destino}/{base}', 'wb') as fd:
                    shutil.copyfileobj(fo, fd)
    def caminho(k):
        a = glob.glob(f'{destino}/*_{k}_v*.shp' if k != 'source' else f'{destino}/*_{k}_v*.dbf')
        assert len(a) == 1, f'{aoi} {k}: {a}'
        return a[0]
    aoi_g = gpd.read_file(caminho('areaOfInterestA')).to_crs(M_CRS)
    peg = gpd.read_file(caminho('imageFootprintA')).to_crs(M_CRS)
    ev = gpd.read_file(caminho('observedEventA')).to_crs(M_CRS)
    fontes = pyogrio.read_dataframe(caminho('source'), read_geometry=False).astype('string').fillna('')
    linhas_fonte = fontes.to_dict('records')
    for r in linhas_fonte:
        FONTES_LINHAS.append({'aoi': aoi, **r})
    sep = M.separar_emsr720([(r, g) for r, g in zip(ev.drop(columns='geometry').to_dict('records'), ev.geometry)],
                            [(r, g) for r, g in zip(peg.drop(columns='geometry').to_dict('records'), peg.geometry)])
    if not M.fonte_do_dia(linhas_fonte, sep['fonte_dano']):
        print(fontes.to_string())
        raise ValueError(f"{aoi}: a fonte {sep['fonte_dano']} do mapeamento do dano não aparece com a data de 06/05/2024 na tabela de fontes acima. Pare e guarde esta saída.")
    EMSR[aoi] = {'aoi': list(aoi_g.geometry), **sep, 'zip': nome_zip}
    print(f"{aoi} {p['nome']}: fonte do dano {sep['fonte_dano']}, {len(sep['pegada'])} pegada(s) dessa fonte, "
          f"{len(sep['nao_analisada'])} parte(s) não analisada(s), {len(sep['agua'])} polígonos de água, "
          f"{len(sep['nao_pontuada'])} de rastro ou deslizamento")

campos = sorted({k for r in FONTES_LINHAS for k in r}, key=lambda k: (k != 'aoi', k))
with open(f'{SAIDA_LOCAL}/emsr720_fontes.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=campos)
    w.writeheader()
    w.writerows(FONTES_LINHAS)
print(f'\nTabela de fontes dos cinco produtos gravada em emsr720_fontes.csv ({len(FONTES_LINHAS)} linhas).')

In [ ]:
# Célula 3 de 4: máscaras dos três tiles no disco local
import rasterio
from shapely.geometry import box
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

CORES = {0: '#c8b27d', 1: '#2166ac', 2: '#000000', 3: '#f2f2f2', 4: '#8c510a', 5: '#d73027',
         6: '#fdd835', 7: '#bdbdbd', 8: '#8f8f8f', 9: '#5f5f5f'}

def gravar_tif(caminho, dados, transform, descricoes):
    perfil = dict(driver='GTiff', height=dados.shape[1], width=dados.shape[2], count=dados.shape[0], dtype='uint8',
                  crs=M_CRS, transform=rasterio.Affine(*transform), tiled=True, blockxsize=512, blockysize=512,
                  compress='deflate')
    with rasterio.open(caminho + '.parcial', 'w', **perfil) as d:
        d.write(dados)
        d.descriptions = descricoes
    os.replace(caminho + '.parcial', caminho)

def ler(caminho, banda=1):
    with rasterio.open(caminho) as s:
        assert (s.height, s.width) == (S.LADO, S.LADO) and str(s.crs) == M_CRS, caminho
        return s.read(banda)

def figura(tile, cod, caminho):
    passo = 10
    cmap = ListedColormap([CORES[k] for k in range(10)])
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(cod[::passo, ::passo], cmap=cmap, norm=BoundaryNorm(np.arange(-0.5, 10.5), 10), interpolation='nearest')
    ax.set_title(f'{tile}: código de cada pixel (1 de cada {passo}), área mapeada da UFRGS a 1 km')
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(handles=[Patch(color=CORES[k], label=f'{k} {M.CODIGOS[k]}') for k in range(10)], loc='upper center',
              bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=8, frameon=False)
    fig.savefig(caminho, dpi=110, bbox_inches='tight')
    plt.close(fig)

def area_km2(m):
    return round(float(m.sum()) * (M.PIXEL ** 2) / 1e6, 4)

ARQUIVOS, JANELAS, TILES_INFO = [], [], {}
t_inicio = time.time()
for tile, g in GRADES.items():
    t0 = time.time()
    T = g['transform']
    forma = (S.LADO, S.LADO)
    print(f'===== {tile}')

    # Entradas do Sentinel-2 e camadas auxiliares, conferidas pelo SHA-256 do registro.
    loc = {}
    for k, rel in g['canonico'].items():
        loc[k] = trazer(f'{S2DIR}/{rel}', SHA_S2[rel])
        ENTRADAS_VERIFICADAS.append({'arquivo': f's2/{rel}', 'sha256': SHA_S2[rel]})
    for k in ('gsw', 'worldcover'):
        a = REG_AUX['arquivos'][f'{tile}_{k}']['canonico']
        loc[k] = trazer(f'{AUXDIR}/{a["arquivo"]}', a['sha256'])
        ENTRADAS_VERIFICADAS.append({'arquivo': f'aux/{a["arquivo"]}', 'sha256': a['sha256']})
    with rasterio.open(loc['s2']) as s:
        assert (s.count, s.height, s.width, str(s.crs)) == (6, S.LADO, S.LADO, M_CRS) and tuple(s.transform)[:6] == T, tile
        sem_dado = np.ones(forma, bool)
        for b in range(1, 7):
            sem_dado &= s.read(b) == 0
        b3, b8a = s.read(2), s.read(4)                      # B2, B3, B4, B8A, B11, B12
    img_agua = b3 > b8a
    del b3, b8a
    os.remove(loc['s2'])
    assert abs(float(sem_dado.mean()) - g['zeros']) < 1e-12, f'{tile}: fração sem dado {sem_dado.mean()}, registro {g["zeros"]}'
    prob, scl = ler(loc['nuvem']), ler(loc['scl'])
    perm = M.permanente(ler(loc['gsw']))
    estr = M.estrato(ler(loc['worldcover']))
    valido = ~sem_dado & (prob < M.NUVEM_PROB) & ~np.isin(scl, M.SCL_EXCLUIDAS) & (scl != 0)

    # UFRGS: mancha na grade com margem de 2 km, distância a ela em pixels.
    Tm = M.transform_com_margem(T)
    forma_m = (S.LADO + 2 * M.MARGEM_PX, S.LADO + 2 * M.MARGEM_PX)
    x0, y1 = Tm[2], Tm[5]
    sel = UFRGS.cx[x0:x0 + forma_m[1] * M.PIXEL, y1 - forma_m[0] * M.PIXEL:y1]
    inund_m = M.rasterizar(list(sel.geometry), Tm, forma_m)
    dist = M.distancia_px(inund_m)
    inund = inund_m[M.MARGEM_PX:-M.MARGEM_PX, M.MARGEM_PX:-M.MARGEM_PX]
    del inund_m
    inv_area = [r.get(f'area_km2_{tile}', '') for r in INV if r['camada'] == CAMADA_UFRGS]
    print(f'UFRGS: {len(sel)} feições perto do tile, mancha {area_km2(inund)} km² no tile '
          f'(inventário, soma dos polígonos: {inv_area[0] if inv_area and inv_area[0] else "não calculada"} km²)')

    # EMSR720: uma camada por produto, todas da mesma referência.
    emsr, emsr_info = [], {}
    caixa = box(T[2], T[5] + S.LADO * T[4], T[2] + S.LADO * T[0], T[5])
    for aoi, e in EMSR.items():
        if not any(gm.intersects(caixa) for gm in e['aoi']):
            continue
        r = {k: M.rasterizar(e[k], T, forma) for k in ('aoi', 'pegada', 'nao_analisada', 'agua', 'nao_pontuada')}
        c = M.camada_emsr720(r['aoi'], r['pegada'], r['nao_analisada'], r['agua'], r['nao_pontuada'], perm)
        if c['coberto'].any():
            emsr.append(c)
            emsr_info[aoi] = {'coberto_km2': area_km2(c['coberto']), 'agua_observada_km2': area_km2(r['agua']),
                              'nao_pontuada_km2': area_km2(r['nao_pontuada']),
                              'agua_observada_fora_da_area_mapeada_px': int((r['agua'] & ~c['coberto']).sum()),
                              'nao_pontuada_fora_da_area_mapeada_px': int((r['nao_pontuada'] & ~c['coberto']).sum())}
            print(f'EMSR720 {aoi}: área mapeada {emsr_info[aoi]["coberto_km2"]} km², água observada {emsr_info[aoi]["agua_observada_km2"]} km², '
                  f'rastro ou deslizamento {emsr_info[aoi]["nao_pontuada_km2"]} km²')
        del r
    sobreposicao = 0
    if len(emsr) > 1:
        soma = np.zeros(forma, np.uint8)
        for c in emsr:
            soma += c['coberto'].astype(np.uint8)
        sobreposicao = int((soma > 1).sum())
        del soma
    if not emsr:
        print('EMSR720: nenhuma área mapeada neste tile.')

    # Conferência de alinhamento (água da imagem, não do modelo).
    alin = {}
    mapeada_1k = M.area_mapeada(dist, 1000)
    testes = [('ufrgs_mancha', inund, valido & mapeada_1k)]
    if emsr:
        cob_e = np.zeros(forma, bool); agua_e = np.zeros(forma, bool)
        for c in emsr:
            cob_e |= c['coberto']; agua_e |= c['agua'] & ~perm
        testes.append(('emsr720_agua_observada', agua_e, valido & cob_e))
    for nome, camada, val in testes:
        res, npx = M.alinhamento(img_agua, camada, val)
        if not res:
            alin[nome] = {'n_pixels': 0}
            continue
        melhor = max(res, key=res.get)
        alin[nome] = {'melhor_deslocamento_linha_coluna': list(melhor), 'concordancia_melhor': round(res[melhor], 4),
                      'concordancia_em_0_0': round(res[(0, 0)], 4), 'n_pixels': npx}
        longe = max(abs(melhor[0]), abs(melhor[1])) > 1 and res[melhor] - res[(0, 0)] > 0.01
        aviso = '   <- longe de (0, 0): pare e guarde esta saída' if longe else ''
        print(f'alinhamento {nome}: melhor {melhor} ({res[melhor]:.4f}), em (0, 0) {res[(0, 0)]:.4f}, {npx:,} pixels{aviso}')
    del img_agua, mapeada_1k, testes

    info = {'fracao_sem_dado': float(sem_dado.mean()), 'ufrgs_feicoes_perto': int(len(sel)),
            'ufrgs_mancha_km2': area_km2(inund), 'ufrgs_mancha_km2_inventario': inv_area[0] if inv_area else '',
            'emsr720_produtos': emsr_info, 'emsr720_sobreposicao_entre_produtos_px': sobreposicao,
            'alinhamento': alin, 'por_distancia': {}}

    for d in M.DISTANCIAS_M:
        u = M.camada_ufrgs(inund, M.area_mapeada(dist, d), perm)
        rot, mot, nrefs = M.consenso([u] + emsr, forma)
        borda = M.borda_de(rot)
        cod, cond = M.codigos(sem_dado, rot, mot, borda, prob, scl)
        del rot, mot, borda
        jan = M.janelas(cod, tile)
        for j in jan:
            JANELAS.append({'distancia_m': d, **j})
        entram = [j for j in jan if j['kept']]
        cont = M.contagens(cod, cond, nrefs, estr)
        camadas_info = {'ufrgs': M.resumo_camada(u)}
        if emsr:
            camadas_info['emsr720'] = {k: sum(M.resumo_camada(c)[k] for c in emsr) for k in ('coberto', 'agua', 'terra', 'nao_pontuado')}
        nome = f'{tile}_classe_{d}m.tif'
        dados = np.stack([cod, nrefs])
        gravar_tif(f'{SAIDA_LOCAL}/{nome}', dados, T, ('codigo', 'n_referencias'))
        ARQUIVOS.append({'arquivo': nome, 'sha256_conteudo': M.sha_conteudo(dados)})
        if d == M.DISTANCIAS_M[0]:
            figura(tile, cod, f'{SAIDA_LOCAL}/{tile}_mascara.png')
        pont = cont['por_codigo']['0'] + cont['por_codigo']['1']
        info['por_distancia'][str(d)] = {
            'camadas': camadas_info, 'contagens': cont,
            'janelas': {'total': len(jan), 'entram': len(entram),
                        'pontuados_nas_que_entram': sum(j['scored_pixels'] for j in entram),
                        'agua_nas_que_entram': sum(j['water_pixels'] for j in entram),
                        'terra_nas_que_entram': sum(j['land_pixels'] for j in entram),
                        'pontuados_perdidos_em_janelas_que_nao_entram': pont - sum(j['scored_pixels'] for j in entram)},
            'arquivo': nome}
        nj = info['por_distancia'][str(d)]['janelas']
        print(f'{d:>5} m: pontuados {pont:>11,} (água {cont["por_codigo"]["1"]:,}, terra {cont["por_codigo"]["0"]:,}); '
              f'janelas {nj["entram"]} de {nj["total"]} entram, com {nj["pontuados_nas_que_entram"]:,} pixels pontuados')
        del u, nrefs, cod, cond, dados
    nome = f'{tile}_estrato.tif'
    gravar_tif(f'{SAIDA_LOCAL}/{nome}', estr[None], T, ('estrato',))
    ARQUIVOS.append({'arquivo': nome, 'sha256_conteudo': M.sha_conteudo(estr[None])})
    info['estrato_contagem'] = {M.ESTRATOS[k]: int((estr == k).sum()) for k in M.ESTRATOS}
    print('códigos, 1 km:', {k: v for k, v in info['por_distancia']['1000']['contagens']['por_codigo'].items() if v})
    TILES_INFO[tile] = info
    for k in ('nuvem', 'scl', 'gsw', 'worldcover'):
        os.remove(loc[k])
    del sem_dado, prob, scl, perm, estr, valido, dist, inund, emsr
    print(f'{tile} pronto em {time.time() - t0:.0f} s.')

for a in ARQUIVOS:
    a['bytes'] = os.path.getsize(f'{SAIDA_LOCAL}/{a["arquivo"]}')
    a['sha256'] = sha256_arquivo(f'{SAIDA_LOCAL}/{a["arquivo"]}')
with open(f'{SAIDA_LOCAL}/rq4_janelas.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(JANELAS[0]))
    w.writeheader()
    w.writerows(JANELAS)
print(f'\nTrês tiles prontos no disco local em {(time.time() - t_inicio) / 60:.0f} min. Nada foi gravado no Drive ainda.')

In [ ]:
# Célula 4 de 4: Drive, conferência e registro
from IPython.display import Image, display

# Copia as máscaras para o Drive. Um arquivo que já existe no Drive nunca é regravado: se o conteúdo for o mesmo,
# fica o do Drive; se for diferente, a célula para.
for a in ARQUIVOS:
    local, no_drive = f'{SAIDA_LOCAL}/{a["arquivo"]}', f'{DESTINO}/{a["arquivo"]}'
    if os.path.exists(no_drive):
        copia = trazer(no_drive, sha256_arquivo(no_drive), pasta=LOCAL)
        with rasterio.open(copia) as s:
            conteudo = M.sha_conteudo(s.read())
        os.remove(copia)
        if conteudo != a['sha256_conteudo']:
            raise ValueError(f'{a["arquivo"]} já existe no Drive com conteúdo diferente do desta rodada. Pare e guarde esta saída.')
        a['no_drive'] = 'ja_existia_com_o_mesmo_conteudo'
        a['bytes'], a['sha256'] = os.path.getsize(no_drive), sha256_arquivo(no_drive)
    else:
        shutil.copyfile(local, no_drive)
        a['no_drive'] = 'gravado_nesta_rodada'
    print(f'{a["arquivo"]}: {a["no_drive"]}')

print('Gravando o Drive e conferindo de novo; pode levar alguns minutos.')
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)
for a in ARQUIVOS:
    caminho = f'{DESTINO}/{a["arquivo"]}'
    esperar(caminho)
    assert sha256_arquivo(caminho) == a['sha256'], f'{a["arquivo"]}: o Drive não guardou o arquivo como esperado. Pare e guarde esta saída.'
print('Todos os arquivos do Drive conferem depois de remontar.')

from importlib.metadata import version
INFO = {'plano': 'prereg/plano_rq4.md, secoes 1 a 3 e mudancas datadas', 'commit_repositorio': commit,
        'mascaras_sha256': SHA_MODULO, 'codigos': {str(k): v for k, v in M.CODIGOS.items()}, 'ordem_dos_motivos': list(M.ORDEM),
        'estratos': {str(k): v for k, v in M.ESTRATOS.items()}, 'distancias_ufrgs_m': list(M.DISTANCIAS_M),
        'parametros': {'gsw_permanente': M.GSW_PERMANENTE, 'nuvem_prob': M.NUVEM_PROB, 'scl_excluidas': list(M.SCL_EXCLUIDAS),
                       'raio_borda': M.RAIO_BORDA, 'janela': M.TAM, 'max_sem_dado': M.MAX_SEM_DADO, 'min_pontuados': M.MIN_PONTUADOS},
        'referencias': {'ufrgs': {'arquivo': NOME_GPKG, 'camada': CAMADA_UFRGS, 'feicoes': int(len(UFRGS))},
                        'emsr720': {aoi: {'zip': e['zip'], 'fonte_dano': e['fonte_dano'], 'pegadas_da_fonte': len(e['pegada']),
                                          'partes_nao_analisadas': len(e['nao_analisada']), 'poligonos_agua': len(e['agua']),
                                          'poligonos_nao_pontuados': len(e['nao_pontuada'])} for aoi, e in EMSR.items()}},
        'entradas_verificadas': ENTRADAS_VERIFICADAS, 'tiles': TILES_INFO, 'arquivos': ARQUIVOS,
        'versoes': {p: version(p) for p in ('numpy', 'scipy', 'rasterio', 'geopandas', 'shapely', 'pyogrio')},
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_LOCAL}/rq4_mascaras.json', 'w'), indent=2, ensure_ascii=False)
with open(f'{SAIDA_LOCAL}/rq4_mascaras_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'bytes', 'sha256', 'sha256_conteudo', 'no_drive'])
    w.writerows([[a['arquivo'], a['bytes'], a['sha256'], a['sha256_conteudo'], a['no_drive']] for a in ARQUIVOS])

os.makedirs(SAIDA_DRIVE, exist_ok=True)
REGISTRO = ['rq4_mascaras.json', 'rq4_mascaras_sha256.csv', 'rq4_janelas.csv', 'emsr720_fontes.csv'] + [f'{t}_mascara.png' for t in GRADES]
for nome in REGISTRO:
    shutil.copyfile(f'{SAIDA_LOCAL}/{nome}', f'{SAIDA_DRIVE}/{nome}')

print()
for tile, info in TILES_INFO.items():
    p = info['por_distancia']['1000']
    c = p['contagens']['por_codigo']
    print(f"{tile}: pontuados {int(c['0']) + int(c['1']):,} (água {int(c['1']):,}); janelas que entram {p['janelas']['entram']}; "
          f"por número de referências {p['contagens']['pontuados_por_nrefs']}")
for tile in GRADES:
    display(Image(f'{SAIDA_LOCAL}/{tile}_mascara.png', width=650))
print(f'Registro em {SAIDA_DRIVE.replace(MEUDRIVE, "MyDrive")}: {REGISTRO}. Nenhum modelo foi rodado.')